# Optimization of Log Loss — Part 2 (Why the Logarithm?)

A guided notebook transcribed and expanded from the Coursera
*Calculus for Machine Learning and Data Science* lecture on **why we use logarithms**
(Week 1 — Derivatives and Optimization).

This notebook explains:

- how the coin example was actually **machine learning** (fitting a model to data)
- **reason 1**: derivatives of **products** are messy, but of **sums** are easy — and the
  logarithm turns products into sums
- **reason 2**: products of many tiny probabilities **underflow**; their logarithms don't
- a practical rule: a complicated product → take the logarithm, especially before
  differentiating

This notebook follows the repository `GUIDELINES.md` template and continues directly from
*Optimization of Log Loss — Part 1*.

## 1. The coin example *was* machine learning

Finding the best coin was exactly the ML workflow:

- **data set:** the 10 flips (7 heads, then 3 tails),
- **model:** a coin that lands heads with probability $P$ (and tails with $1 - P$),
- **fitting:** find the model most likely to produce the data, by **minimizing the log
  loss** — which gave the optimal $P = 0.7$.

So why introduce a logarithm at all? Two reasons.

## 2. Reason 1 — products are hard to differentiate; sums are easy

The likelihood is a **product**, $G(P) = P^7 (1-P)^3$. Differentiating a product of *many*
factors means iterating the product rule — it gets messy fast. The logarithm converts the
product into a **sum**:

$$
\log G(P) = 7\log P + 3\log(1-P),
$$

whose derivative is clean (just $\tfrac1P$-type terms):

$$
\frac{d}{dP}\log G = \frac{7}{P} - \frac{3}{1-P}.
$$

The denominators are a small price to pay for avoiding the ugly product-rule expression.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

P = sp.symbols('P', positive=True)
G = P**7 * (1 - P)**3

# direct derivative of the product (messy) vs derivative of the log (clean)
print("direct  G'(P)      =", sp.diff(G, P))
print("expanded           =", sp.expand(sp.diff(G, P)))
print("\nlog version d/dP logG =", sp.simplify(sp.diff(7*sp.log(P) + 3*sp.log(1-P), P)))

# both give the same optimum
print("\noptimum from G'      :", [s for s in sp.solve(sp.diff(G, P), P)])
print("optimum from log     :", sp.solve(sp.diff(7*sp.log(P)+3*sp.log(1-P), P), P))

direct  G'(P)      = -3*P**7*(1 - P)**2 + 7*P**6*(1 - P)**3
expanded           = -10*P**9 + 27*P**8 - 24*P**7 + 7*P**6

log version d/dP logG = (10*P - 7)/(P*(P - 1))

optimum from G'      : [7/10, 1]
optimum from log     : [7/10]


### The mess grows with more factors

For a product of many terms, the direct derivative explodes in length, while the log version
stays a simple **sum of fractions**.

In [2]:
# a product of several different powers -> compare derivative sizes
expr = P**3 * (1-P)**2 * (P + sp.Rational(1,2))**4 * (2 - P)**3
direct = sp.diff(expr, P)
log_form = sp.diff(sp.log(expr), P)   # = sum of term-by-term log derivatives

print("length of direct derivative expression :", len(str(sp.expand(direct))))
print("length of log-derivative expression     :", len(str(sp.simplify(log_form))))
print("\nlog-derivative (clean sum of fractions):")
sp.pprint(sp.simplify(log_form))

length of direct derivative expression : 110
length of log-derivative expression     : 59

log-derivative (clean sum of fractions):
  ⎛   3      2          ⎞
6⋅⎝4⋅P  - 8⋅P  + 2⋅P + 1⎠
─────────────────────────
   ⎛   3      2        ⎞ 
 P⋅⎝2⋅P  - 5⋅P  + P + 2⎠ 


## 3. Reason 2 — products of tiny numbers underflow

A probability that is a product of **many** numbers between $0$ and $1$ becomes
**extraordinarily small** — small enough that a computer rounds it to $0$ (*underflow*).

The logarithm fixes this: $\log$ of a tiny positive number is a **large negative** number,
which computers represent comfortably. And a product becomes a **sum of logs**:

$$
\log\!\Big(\prod_i p_i\Big) = \sum_i \log p_i .
$$

In [3]:
rng = np.random.default_rng(0)
probs = rng.uniform(0.01, 0.5, size=2000)   # 2000 small probabilities

product = np.prod(probs)
log_sum = np.sum(np.log(probs))

print(f"direct product of 2000 probabilities : {product}")          # underflows to 0.0
print(f"sum of their logarithms              : {log_sum:.2f}")      # a fine large-negative number
print(f"exp(log_sum) (would-be product)      : {np.exp(log_sum)}")  # also 0.0 — confirms underflow
print("\n-> the raw product underflows to 0, but the log sum is perfectly usable.")

direct product of 2000 probabilities : 0.0
sum of their logarithms              : -3249.10
exp(log_sum) (would-be product)      : 0.0

-> the raw product underflows to 0, but the log sum is perfectly usable.


## 4. The practical rule

> *"Any time in machine learning that you have a very complicated product, think of using
> the logarithm. It may simplify it a lot, especially if you're taking derivatives."*

Two payoffs:

1. **Calculus:** $\log$ turns products into sums, so derivatives become simple.
2. **Numerics:** $\log$ turns tiny products into manageable sums, avoiding underflow.

This is why losses in ML are almost always written as **log-likelihoods** (the log loss).

## 5. Exercises

### Basic
1. In the coin example, what was the data set, the model, and what was minimized?
2. Give two reasons for using logarithms in optimization.

### Intermediate
3. Rewrite $\log\big(P^4(1-P)^6\big)$ as a sum, then differentiate it.
4. Compute the product of 1000 values equal to $0.1$ directly and via logs. What happens?

### Advanced
5. Show that maximizing $\prod_i p_i$ and maximizing $\sum_i \log p_i$ give the same
   optimum.
6. Explain why log loss (negative log-likelihood) is numerically preferable to the raw
   likelihood for large data sets.

## 6. Solutions / Checks

In [4]:
P = sp.symbols('P', positive=True)

# 3.
expr = sp.log(P**4 * (1-P)**6)
print("3. log form:", sp.expand_log(expr, force=True))
print("   derivative:", sp.simplify(sp.diff(expr, P)))

# 4.
direct = 0.1**1000
logsum = 1000*np.log(0.1)
print(f"4. 0.1^1000 direct = {direct} (underflow);  via logs = {logsum:.1f} (fine)")

print("\n1. Data = the 10 flips (7H,3T); model = a coin with P(H)=P; we minimized the log loss.")
print("2. (i) products are hard to differentiate, logs make them sums; (ii) tiny products "
      "underflow, logs don't.")
print("5. log is strictly increasing, so it preserves the location of the maximum.")
print("6. Multiplying many probabilities underflows; summing their logs stays representable.")

3. log form: 4*log(P) + 6*log(1 - P)
   derivative: 2*(5*P - 2)/(P*(P - 1))
4. 0.1^1000 direct = 0.0 (underflow);  via logs = -2302.6 (fine)

1. Data = the 10 flips (7H,3T); model = a coin with P(H)=P; we minimized the log loss.
2. (i) products are hard to differentiate, logs make them sums; (ii) tiny products underflow, logs don't.
5. log is strictly increasing, so it preserves the location of the maximum.
6. Multiplying many probabilities underflows; summing their logs stays representable.


## 7. Conclusion

- The coin task was machine learning: fit a model (coin $P$) to data (the flips) by
  minimizing the **log loss**, giving $P = 0.7$.
- We use the **logarithm** for two reasons:
  1. it converts **products into sums**, making derivatives easy;
  2. it converts **tiny products into manageable sums**, avoiding numerical **underflow**.
- Rule of thumb: a complicated product → take the log, especially before differentiating —
  which is exactly why ML uses log-likelihoods / the log loss.

## Appendix — Source transcript

Transcribed with OpenAI Whisper (`small.en`) from a locally downloaded video
(`~/Downloads/index (26).mp4`), the Coursera *Calculus for Machine Learning and Data
Science* (Week 1) lecture **optimization of log loss — part 2**. Key spoken points, lightly
cleaned:

> The previous coin example was exactly machine learning: finding the best model for a data
> set. The data set was the ten flips (seven heads then three tails), and the model was a
> coin landing heads with probability $P$ and tails with $1-P$; you found the model that
> most likely fits the data by minimizing the log loss, getting $P = 0.7$. Why a logarithm?
> First, derivatives of products are hard — the product rule gets messy for many factors —
> while derivatives of sums are easy, and the logarithm turns the product into a sum, giving
> a clean derivative with simple $1/P$ denominators. Second, the product of many tiny things
> is tiny: a product of 1000 probabilities between 0 and 1 can be so small a computer can't
> handle it, but its logarithm is a large negative number computers handle fine. So whenever
> you have a complicated product in machine learning, consider the logarithm — it often
> simplifies things, especially with derivatives.